# Validation Split Group Diagnostics

This notebook compares the validation set to the training set to check whether our validation split is strong enough for model selection.

The main question is simple: **does validation look like the data we train on, while still testing meaningful generalization?**

We inspect the split visually across the groups that matter for this challenge:

- gender, because the official metric compares female and male errors;
- occlusion level, because high occlusion is rare but heavily weighted;
- source database, because the three databases have different visual distributions;
- identity-like `group_id`, because database3 contains repeated people.

The default project split is the fixed row-level stratified split. It is designed for leaderboard-oriented experiments. The optional group-level split is used as a robustness check for unseen identity-like groups.


## 1. Setup

The notebook is intentionally self-contained. It reads the training CSV and the saved split files, then rebuilds the derived metadata used by the pipeline.


In [ ]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find the repository root.")


REPO_ROOT = find_repo_root()
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

add_path_metadata = importlib.import_module("face_occlusion.data.metadata").add_path_metadata

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.dpi"] = 120

DATA_DIR = REPO_ROOT / "data"
TRAIN_CSV = DATA_DIR / "occlusion_datasets" / "train.csv"
ROW_SPLIT_CSV = REPO_ROOT / "outputs" / "splits" / "canonical_row_split.csv"
GROUP_SPLIT_CSV = REPO_ROOT / "outputs" / "splits" / "group_robustness_split.csv"

OCCLUSION_BINS = [0.0, 0.05, 0.10, 0.20, 0.40, 0.60, 1.0]
OCCLUSION_LABELS = [
    "0-0.05",
    "0.05-0.10",
    "0.10-0.20",
    "0.20-0.40",
    "0.40-0.60",
    "0.60-1.00",
]

GENDER_LABELS = {0: "female", 1: "male"}

print(f"Repository root: {REPO_ROOT}")
print(f"Train CSV: {TRAIN_CSV.relative_to(REPO_ROOT)}")
print(f"Row split: {ROW_SPLIT_CSV.relative_to(REPO_ROOT)}")
print(f"Group split: {GROUP_SPLIT_CSV.relative_to(REPO_ROOT)}")

## 2. Load the split

A good validation split must be reproducible. We therefore load the saved split file instead of creating a new random split inside the notebook.


In [ ]:
def add_analysis_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["gender_label"] = out["gender"].map(GENDER_LABELS).fillna("unknown")
    out["occlusion_bin"] = pd.cut(
        out["target"],
        bins=OCCLUSION_BINS,
        labels=OCCLUSION_LABELS,
        include_lowest=True,
    )
    out["face_id_label"] = np.where(
        out["face_id"].astype(int) == 0,
        "FaceId-0",
        "FaceId != 0",
    )
    return out


def load_split_dataset(split_csv: Path) -> pd.DataFrame:
    train = pd.read_csv(TRAIN_CSV)
    train = train.rename(columns={"FaceOcclusion": "target"})
    train = add_path_metadata(train)

    split = pd.read_csv(split_csv)
    if not {"filename", "split"}.issubset(split.columns):
        raise ValueError(f"{split_csv} must contain filename and split columns.")

    # Merge only the split assignment. Metadata is rebuilt from the training CSV so this
    # notebook also works with minimal split files containing only filename/split.
    df = train.merge(split[["filename", "split"]], on="filename", how="inner")
    missing_rows = len(train) - len(df)
    if missing_rows:
        print(f"Warning: {missing_rows:,} training rows were not found in {split_csv.name}.")
    return add_analysis_columns(df)


row_df = load_split_dataset(ROW_SPLIT_CSV)
group_df = load_split_dataset(GROUP_SPLIT_CSV) if GROUP_SPLIT_CSV.exists() else None

print(f"Canonical row split rows: {len(row_df):,}")
display(row_df.head())

## 3. Train vs validation at a glance

In this notebook, `train` means the training portion of the saved split and `val` means the held-out validation portion. The goal is to verify that validation is representative of training on the variables that matter for the challenge.

The first plot compares the full target distribution. The second plot shows validation-minus-training proportion gaps for the main categorical groups. Values close to zero mean validation mirrors training.


In [ ]:
def train_val_proportions(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    counts = df.groupby(["split", group_col], observed=False).size().reset_index(name="count")
    counts["proportion"] = counts["count"] / counts.groupby("split")["count"].transform("sum")
    return counts


def distribution_gap(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    prop = train_val_proportions(df, group_col)
    pivot = prop.pivot(index=group_col, columns="split", values="proportion").fillna(0)
    if "train" not in pivot:
        pivot["train"] = 0.0
    if "val" not in pivot:
        pivot["val"] = 0.0
    gap = pivot.assign(gap=pivot["val"] - pivot["train"])
    gap = gap.reset_index().rename(columns={group_col: "value"})
    gap.insert(0, "group", group_col)
    return gap


gap_df = pd.concat(
    [
        distribution_gap(row_df, "gender_label"),
        distribution_gap(row_df, "database"),
        distribution_gap(row_df, "occlusion_bin"),
        distribution_gap(row_df, "face_id_label"),
    ],
    ignore_index=True,
)
gap_df["label"] = gap_df["group"] + " = " + gap_df["value"].astype(str)
gap_df = gap_df.sort_values("gap")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.ecdfplot(data=row_df, x="target", hue="split", ax=axes[0])
axes[0].set_title("Target ECDF: training vs validation")
axes[0].set_xlabel("FaceOcclusion")
axes[0].set_ylabel("Cumulative proportion")

sns.barplot(data=gap_df, y="label", x="gap", hue="group", dodge=False, ax=axes[1])
axes[1].axvline(0, color="black", linewidth=1)
axes[1].set_title("Validation - training proportion gaps")
axes[1].set_xlabel("Proportion gap")
axes[1].set_ylabel("")
axes[1].xaxis.set_major_formatter(lambda value, _: f"{value:+.1%}")
axes[1].legend(title="Group", loc="lower right")

plt.tight_layout()
plt.show()

**Interpretation.** This is the quickest robustness check. If the ECDF curves overlap and the proportion gaps stay close to zero, the validation set is representative of the training set for the variables we intentionally stratified on.


## 4. Split size

Before looking at detailed distributions, we first check whether the split has the expected train/validation size. A strongly imbalanced validation size would make metrics noisy.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
split_counts = row_df["split"].value_counts().rename_axis("split").reset_index(name="count")
sns.barplot(
    data=split_counts,
    x="split",
    y="count",
    hue="split",
    palette="Set2",
    ax=ax,
    legend=False,
)
ax.set_title("Canonical row split size")
ax.set_xlabel("")
ax.set_ylabel("Rows")
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f")
plt.show()

val_ratio = (row_df["split"] == "val").mean()
print(f"Validation ratio: {val_ratio:.1%}")

**Interpretation.** We expect validation to be about 20% of the labeled data. This is large enough to make subgroup diagnostics meaningful while leaving most examples for training.


## 5. Target distribution

The target distribution is the most important visual check. The challenge has many low occlusion examples and relatively few high occlusion examples. Validation should preserve that shape; otherwise, the score can become misleading.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(
    data=row_df,
    x="target",
    hue="split",
    stat="density",
    common_norm=False,
    bins=50,
    element="step",
    ax=axes[0],
)
axes[0].set_title("Target density by split")
axes[0].set_xlabel("FaceOcclusion")

sns.boxplot(
    data=row_df,
    x="split",
    y="target",
    hue="split",
    palette="Set2",
    ax=axes[1],
    legend=False,
)
axes[1].set_title("Target spread by split")
axes[1].set_xlabel("")
axes[1].set_ylabel("FaceOcclusion")

plt.tight_layout()
plt.show()

**Interpretation.** The two curves should nearly overlap. If validation has too many or too few high-occlusion images, leaderboard comparisons between experiments become less stable.


## 6. Gender and database balance

The official score is gender-aware, and the databases behave like hidden domains. This section checks whether validation preserves both distributions.


In [ ]:
def proportion_by_split(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    counts = df.groupby(["split", group_col], observed=False).size().reset_index(name="count")
    counts["proportion"] = counts["count"] / counts.groupby("split")["count"].transform("sum")
    return counts


def plot_split_proportions(
    df: pd.DataFrame,
    group_col: str,
    title: str,
    ax: plt.Axes,
    rotation: int = 0,
) -> None:
    plot_df = proportion_by_split(df, group_col)
    sns.barplot(data=plot_df, x=group_col, y="proportion", hue="split", ax=ax)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel("Proportion")
    ax.tick_params(axis="x", rotation=rotation)
    ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")


fig, axes = plt.subplots(1, 2, figsize=(13, 4))
plot_split_proportions(row_df, "gender_label", "Gender distribution", axes[0])
plot_split_proportions(row_df, "database", "Database distribution", axes[1])
plt.tight_layout()
plt.show()

**Interpretation.** Strong train/validation alignment here means the validation score is not dominated by a different gender mix or a different source database mix. The database plot is especially important because database3 dominates the dataset.


## 7. Occlusion bins

The split is stratified on occlusion bins. This visual check makes sure rare medium/high occlusion examples are still represented in validation.


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
plot_split_proportions(row_df, "occlusion_bin", "Occlusion-bin distribution", ax, rotation=30)
plt.tight_layout()
plt.show()

**Interpretation.** The high-occlusion bins are small but important. If they disappear from validation, we cannot trust the validation score to judge high-occlusion behavior.


## 8. Joint stratification checks

A split can look balanced one column at a time and still fail for combinations. The heatmaps below show whether validation keeps the same structure for `gender × occlusion_bin` and `database × occlusion_bin`.


In [ ]:
def joint_proportion_matrix(
    df: pd.DataFrame,
    split_name: str,
    index: str,
    columns: str,
) -> pd.DataFrame:
    subset = df[df["split"] == split_name]
    matrix = pd.crosstab(subset[index], subset[columns], normalize="all")
    index_values = sorted(df[index].dropna().unique())
    if hasattr(df[columns], "cat"):
        column_values = list(df[columns].cat.categories)
    else:
        column_values = sorted(df[columns].dropna().unique())
    return matrix.reindex(index=index_values, columns=column_values, fill_value=0)


def plot_joint_heatmaps(df: pd.DataFrame, index: str, columns: str, title: str) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    train_matrix = joint_proportion_matrix(df, "train", index, columns)
    val_matrix = joint_proportion_matrix(df, "val", index, columns)
    diff_matrix = (val_matrix - train_matrix).reindex_like(train_matrix).fillna(0)

    sns.heatmap(train_matrix, cmap="Blues", annot=True, fmt=".1%", ax=axes[0])
    axes[0].set_title("Train")
    sns.heatmap(val_matrix, cmap="Blues", annot=True, fmt=".1%", ax=axes[1])
    axes[1].set_title("Validation")
    sns.heatmap(diff_matrix, cmap="coolwarm", center=0, annot=True, fmt="+.1%", ax=axes[2])
    axes[2].set_title("Validation - train")
    fig.suptitle(title, y=1.05)
    plt.tight_layout()
    plt.show()


plot_joint_heatmaps(row_df, "gender_label", "occlusion_bin", "Gender × occlusion-bin balance")
plot_joint_heatmaps(row_df, "database", "occlusion_bin", "Database × occlusion-bin balance")

**Interpretation.** The difference heatmap is the main panel to read. Values close to zero mean validation mirrors training for that subgroup. Large red or blue cells would flag a weak validation split for that subgroup.


## 9. Target by database and gender

Now we check whether subgroup target distributions remain comparable. This matters because a database or gender group can have the same count but a different occlusion severity profile.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.violinplot(
    data=row_df,
    x="database",
    y="target",
    hue="split",
    split=True,
    inner="quart",
    ax=axes[0],
)
axes[0].set_title("Target distribution by database")
axes[0].set_xlabel("")
axes[0].set_ylabel("FaceOcclusion")

sns.violinplot(
    data=row_df,
    x="gender_label",
    y="target",
    hue="split",
    split=True,
    inner="quart",
    ax=axes[1],
)
axes[1].set_title("Target distribution by gender")
axes[1].set_xlabel("")
axes[1].set_ylabel("FaceOcclusion")

plt.tight_layout()
plt.show()

**Interpretation.** Similar violin shapes mean validation is testing the same kind of occlusion examples seen during training. Mismatched tails would be a warning, especially in the high-target region.


## 10. Identity-like group overlap

The canonical split is row-level, so database3 identities can appear in both train and validation. That is intentional for leaderboard-oriented validation, because the challenge test set also contains many database3 identities seen in train.

However, this also means the row split does **not** fully measure generalization to unseen identity-like groups. The optional group split should be used for that question.


In [ ]:
def group_overlap_summary(df: pd.DataFrame, label: str) -> dict[str, object]:
    train_groups = set(df.loc[df["split"] == "train", "group_id"].astype(str))
    val_groups = set(df.loc[df["split"] == "val", "group_id"].astype(str))
    shared_groups = train_groups & val_groups
    val_rows_seen = df.loc[df["split"] == "val", "group_id"].astype(str).isin(train_groups)
    return {
        "split": label,
        "train_groups": len(train_groups),
        "val_groups": len(val_groups),
        "shared_groups": len(shared_groups),
        "val_rows_seen_ratio": val_rows_seen.mean(),
    }


summaries = [group_overlap_summary(row_df, "row_stratified")]
if group_df is not None:
    summaries.append(group_overlap_summary(group_df, "group_stratified"))
overlap_df = pd.DataFrame(summaries)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
long_counts = overlap_df.melt(
    id_vars="split",
    value_vars=["train_groups", "val_groups", "shared_groups"],
    var_name="group_type",
    value_name="count",
)
sns.barplot(data=long_counts, x="split", y="count", hue="group_type", ax=axes[0])
axes[0].set_title("Unique group counts")
axes[0].set_xlabel("")
axes[0].set_ylabel("Groups")
axes[0].tick_params(axis="x", rotation=15)

sns.barplot(
    data=overlap_df,
    x="split",
    y="val_rows_seen_ratio",
    hue="split",
    palette="Set2",
    ax=axes[1],
    legend=False,
)
axes[1].set_title("Validation rows with groups seen in train")
axes[1].set_xlabel("")
axes[1].set_ylabel("Validation row ratio")
axes[1].yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

display(overlap_df)

**Interpretation.** High overlap in the row split is not a bug; it tells us the split is close to the leaderboard setting. Low or zero overlap in the group split is what we want for the robustness check. Together, the two splits answer different questions.


## 11. Are repeated groups dominating validation?

Even if the split is balanced by rows, a few repeated identities could dominate validation. The plot below checks whether validation depends too heavily on a small number of groups.


In [ ]:
def group_size_frame(df: pd.DataFrame) -> pd.DataFrame:
    return (
        df.groupby(["split", "group_id"])
        .size()
        .reset_index(name="rows_per_group")
        .sort_values("rows_per_group", ascending=False)
    )


group_sizes = group_size_frame(row_df)
top_val_groups = group_sizes[group_sizes["split"] == "val"].head(20)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(
    data=group_sizes,
    x="rows_per_group",
    hue="split",
    bins=40,
    stat="density",
    common_norm=False,
    ax=axes[0],
)
axes[0].set_title("Rows per identity-like group")
axes[0].set_xlabel("Rows per group")
axes[0].set_ylabel("Density")

sns.barplot(data=top_val_groups, y="group_id", x="rows_per_group", color="#4C78A8", ax=axes[1])
axes[1].set_title("Top validation groups by row count")
axes[1].set_xlabel("Validation rows")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

**Interpretation.** A long tail is expected for database3. The useful check is whether a few groups have unusually many validation rows. If so, their behavior can move the validation score more than we would like.


## 12. FaceId and source-folder checks

Most database3 images use `FaceId-0`, but non-zero face IDs exist. We do not use this as a model feature, but it is useful for checking whether rare face detections are present in validation.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_split_proportions(row_df, "face_id_label", "FaceId-0 vs non-zero FaceId", axes[0])

top_sources = row_df["source_subfolder"].value_counts().head(12).index
source_df = row_df[row_df["source_subfolder"].isin(top_sources)].copy()
source_prop = proportion_by_split(source_df, "source_subfolder")
sns.barplot(
    data=source_prop,
    y="source_subfolder",
    x="proportion",
    hue="split",
    ax=axes[1],
)
axes[1].set_title("Top source folders")
axes[1].set_xlabel("Proportion inside displayed sources")
axes[1].set_ylabel("")
axes[1].xaxis.set_major_formatter(lambda value, _: f"{value:.0%}")

plt.tight_layout()
plt.show()

**Interpretation.** This is a sanity check for rare path-derived metadata. Large differences would suggest that validation has unusual face detections or source folders compared with training.


## 13. Compact reliability summary

The table below gives a small numeric companion to the plots. The values are not a replacement for the visuals; they simply highlight the largest distribution gaps.


In [ ]:
def max_proportion_gap(df: pd.DataFrame, group_col: str) -> float:
    prop = proportion_by_split(df, group_col)
    pivot = prop.pivot(index=group_col, columns="split", values="proportion").fillna(0)
    return float((pivot.get("val", 0) - pivot.get("train", 0)).abs().max())


def target_mean_gap(df: pd.DataFrame) -> float:
    means = df.groupby("split")["target"].mean()
    return float(abs(means.get("val", np.nan) - means.get("train", np.nan)))


row_overlap = group_overlap_summary(row_df, "row_stratified")
summary = pd.DataFrame(
    [
        {"check": "Validation ratio", "value": f"{(row_df['split'] == 'val').mean():.1%}"},
        {"check": "Target mean gap", "value": f"{target_mean_gap(row_df):.4f}"},
        {
            "check": "Max gender proportion gap",
            "value": f"{max_proportion_gap(row_df, 'gender_label'):.2%}",
        },
        {
            "check": "Max database proportion gap",
            "value": f"{max_proportion_gap(row_df, 'database'):.2%}",
        },
        {
            "check": "Max occlusion-bin proportion gap",
            "value": f"{max_proportion_gap(row_df, 'occlusion_bin'):.2%}",
        },
        {"check": "Shared row-split groups", "value": f"{row_overlap['shared_groups']:,}"},
        {
            "check": "Row-split val rows seen in train",
            "value": f"{row_overlap['val_rows_seen_ratio']:.1%}",
        },
    ]
)
display(summary)

**Current saved-split reading.** With the current canonical row split, validation is very well aligned with training on the intended stratification axes: the validation ratio is 20%, the target mean gap is about 0.0001, and the largest gender/database/occlusion-bin proportion gaps are near zero.

The important limitation is identity overlap: most row-split validation rows belong to identity-like groups also seen in training. This supports the row split as our main leaderboard-oriented validation, but it should not be treated as a pure unseen-identity test. The group split remains necessary for robustness checks.


## 14. Decision guide

Use the plots above to read the split in two layers:

1. **Distribution reliability.** If target, gender, database, and occlusion-bin plots are well aligned, the canonical row split is reliable for comparing leaderboard-oriented experiments.
2. **Identity robustness.** If row-level validation contains many groups also seen in train, it does not fully test unseen-identity generalization. That is why the optional group-level split must remain available.

For this project, the intended workflow is therefore:

- train most experiments once on the fixed `row_stratified` split;
- use the `group_stratified` split selectively as a robustness check;
- compare models only when they use the same split;
- before final submission, consider retraining the selected model on all labeled data.
